# 🦾 Project 4.2 — Servo Command Tape
**DSA for Mechatronics · Week 04 — Linked Lists**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A servo controller stores a **command tape** — a sequence of target angles recorded during a teaching pass.
During playback the tape can run **forward** (repeat the motion) or **backward** (reverse the motion).
It is modelled as a **doubly linked list** so both directions are O(n) and any individual command can be edited or removed in O(1) once located.


## Step 1 — Define DNode and generate your command tape

In [ ]:
class DNode:
    """One command frame in the servo tape."""
    def __init__(self, frame, angle, speed):
        self.frame = frame      # frame number (time index)
        self.angle = angle      # target angle in degrees
        self.speed = speed      # degrees/second
        self.prev  = None
        self.next  = None
    def __repr__(self):
        return f"F{self.frame}({self.angle}°@{self.speed}°/s)"

# ── Personalised parameters ─────────────────────────────────────────
N_FRAMES    = random.randint(14, 22)
ANGLE_MIN   = random.randint(-60, -20)
ANGLE_MAX   = random.randint( 40,  90)
SPEED_MIN   = random.randint( 10,  30)
SPEED_MAX   = random.randint( 80, 150)
JITTER_PROB = round(random.uniform(0.10, 0.25), 2)   # fraction of jitter frames

# Build doubly linked list
angles = [round(ANGLE_MIN + (ANGLE_MAX - ANGLE_MIN) * i / (N_FRAMES - 1)
                + random.gauss(0, 4), 1)
          for i in range(N_FRAMES)]
speeds = [random.randint(SPEED_MIN, SPEED_MAX) for _ in range(N_FRAMES)]

head = tail = None
for i in range(N_FRAMES):
    node = DNode(i, angles[i], speeds[i])
    if head is None:
        head = tail = node
    else:
        tail.next  = node
        node.prev  = tail
        tail       = node

def tape_to_list(h, forward=True):
    out = []
    cur = h if forward else tail   # start from tail for reverse
    while cur:
        out.append(cur)
        cur = cur.next if forward else cur.prev
    return out

print(f"Servo tape parameters:")
print(f"  Frames       : {N_FRAMES}")
print(f"  Angle range  : {ANGLE_MIN}° → {ANGLE_MAX}°")
print(f"  Speed range  : {SPEED_MIN} – {SPEED_MAX} °/s")
print()
print(f"  {'Frame':>6}  {'Angle':>8}  {'Speed':>8}")
print(f"  {'─'*6}  {'─'*8}  {'─'*8}")
for n in tape_to_list(head):
    print(f"  {n.frame:6d}  {n.angle:>7.1f}°  {n.speed:>6d} °/s")


## Step 2 — Bidirectional traversal: forward and reverse playback time

In [ ]:
def playback_time(head_node, forward=True):
    """
    Compute total playback time by traversing the tape.
    Time for each frame = abs(angle_change) / speed
    Traverse forward (head→tail) or backward (tail→head).
    Returns total seconds and list of (frame, delta_time).
    """
    nodes   = tape_to_list(head_node, forward=forward)
    total   = 0.0
    details = []
    for i in range(1, len(nodes)):
        a_from = nodes[i-1].angle
        a_to   = nodes[i].angle
        spd    = nodes[i].speed if forward else nodes[i-1].speed
        dt     = abs(a_to - a_from) / spd
        total += dt
        details.append((nodes[i].frame, round(dt, 3)))
    return round(total, 3), details

fwd_time, fwd_detail = playback_time(head, forward=True)
rev_time, rev_detail = playback_time(head, forward=False)

print(f"Forward playback  : {fwd_time:.3f} s")
print(f"Backward playback : {rev_time:.3f} s")
print(f"  (times differ because speed values are frame-local)")
print()
print("Slowest 3 transitions (forward):")
top3 = sorted(fwd_detail, key=lambda x: -x[1])[:3]
for frame, dt in top3:
    print(f"  → frame {frame:3d}: {dt:.3f} s")


## Step 3 — O(1) node deletion: remove jitter frames (doubly linked advantage)

In [ ]:
def find_jitter_frames(head_node, window=3, max_deviation=8.0):
    """
    A frame is 'jitter' if its angle deviates more than max_deviation
    from the local window average of its neighbours.
    Returns list of DNode references.
    """
    nodes   = tape_to_list(head_node)
    jitter  = []
    for i in range(1, len(nodes) - 1):   # skip first and last
        neighbours = nodes[max(0, i-window) : i] + nodes[i+1 : i+1+window]
        avg = sum(n.angle for n in neighbours) / len(neighbours)
        if abs(nodes[i].angle - avg) > max_deviation:
            jitter.append(nodes[i])
    return jitter

def delete_node(node, head_ref, tail_ref):
    """
    Delete a node from a doubly linked list in O(1).
    O(1) is only possible because we have prev pointers.
    Returns updated (head, tail).
    """
    if node.prev:
        node.prev.next = node.next
    else:
        head_ref = node.next     # deleted the head

    if node.next:
        node.next.prev = node.prev
    else:
        tail_ref = node.prev     # deleted the tail

    node.next = node.prev = None
    return head_ref, tail_ref

jitter_nodes = find_jitter_frames(head)
n_jitter     = len(jitter_nodes)
jitter_frames = [n.frame for n in jitter_nodes]

for node in jitter_nodes:
    head, tail = delete_node(node, head, tail)

remaining = tape_to_list(head)

print(f"Jitter detection and removal (window=3, max_dev=8.0°):")
print(f"  Jitter frames found   : {n_jitter}")
print(f"  Jitter frame numbers  : {jitter_frames}")
print(f"  Frames after cleaning : {len(remaining)}")


## Step 4 — Palindrome check: is the angle sequence symmetric?

In [ ]:
def is_angle_palindrome(head_node, tolerance=5.0):
    """
    Check if the tape is 'symmetric' — i.e. the angle sequence reads the
    same forward and backward within a tolerance.
    Uses fast/slow to find middle, then compares outward from centre.
    """
    nodes = tape_to_list(head_node)
    n     = len(nodes)
    i, j  = 0, n - 1
    mismatches = []
    while i < j:
        diff = abs(nodes[i].angle - nodes[j].angle)
        if diff > tolerance:
            mismatches.append((nodes[i].frame, nodes[j].frame, round(diff, 1)))
        i += 1
        j -= 1
    return len(mismatches) == 0, mismatches

symmetric, mismatches = is_angle_palindrome(head)

print(f"Symmetry check (tolerance ±5.0°):")
print(f"  Symmetric motion : {'YES' if symmetric else 'NO'}")
if mismatches:
    print(f"  Asymmetric pairs : {len(mismatches)}")
    for fa, fb, diff in mismatches[:4]:
        print(f"    frame {fa:3d} vs frame {fb:3d}  Δ = {diff}°")
else:
    print("  All frame pairs within tolerance ✅")


## Step 5 — In-place reversal: generate the reverse teaching tape

In [ ]:
def reverse_doubly(head_node):
    """
    Reverse a doubly linked list in-place by swapping next/prev on every node.
    O(n) time, O(1) space.
    Returns new head (which was the old tail).
    """
    current = head_node
    new_head = None
    while current:
        # swap prev and next
        current.prev, current.next = current.next, current.prev
        new_head = current           # keep track of last visited (will be new head)
        current  = current.prev      # move to "next" (now stored in prev after swap)
    return new_head

original_order = [n.frame for n in tape_to_list(head)]
head = reverse_doubly(head)
tail = head
while tail.next:   # update tail pointer after reversal
    tail = tail.next
reversed_order = [n.frame for n in tape_to_list(head)]

print("Tape reversed (reverse teaching pass):")
print(f"  Original : {original_order}")
print(f"  Reversed : {reversed_order}")
print(f"  Head frame: {head.frame}  Tail frame: {tail.frame}")


## 📋 Final Report

In [ ]:
W = 58
n_clean     = len(remaining)
time_ratio  = round(rev_time / fwd_time, 3) if fwd_time else 0

print("╔" + "═"*W + "╗")
print("║" + " SERVO COMMAND TAPE — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  TAPE PARAMETERS" + " "*(W-17) + "║")
print(f"║  {'Frames recorded':<28}: {N_FRAMES:<26}║")
print(f"║  {'Angle range':<28}: {ANGLE_MIN}° → {ANGLE_MAX}°{'':<20}║")
print(f"║  {'Speed range':<28}: {SPEED_MIN}–{SPEED_MAX} °/s{'':<20}║")
print("╠" + "═"*W + "╣")
print("║  PLAYBACK ANALYSIS" + " "*(W-19) + "║")
print(f"║  {'Forward playback time':<28}: {fwd_time:.3f} s{'':<21}║")
print(f"║  {'Backward playback time':<28}: {rev_time:.3f} s{'':<21}║")
print(f"║  {'Rev / Fwd time ratio':<28}: {time_ratio:<26}║")
print("╠" + "═"*W + "╣")
print("║  CLEANING & ANALYSIS" + " "*(W-21) + "║")
print(f"║  {'Jitter frames removed':<28}: {n_jitter:<26}║")
print(f"║  {'Jitter frame numbers':<28}: {str(jitter_frames):<26}║")
print(f"║  {'Frames after cleaning':<28}: {n_clean:<26}║")
print(f"║  {'Symmetric motion':<28}: {'YES' if symmetric else 'NO ('+str(len(mismatches))+' asymmetric pairs)':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which linked list concept did you find most important, and why?**
Pick one technique from the notebook (fast/slow pointers, dummy head, reversal, cycle detection…) and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
